# Data Preparation Pipeline
## Pneumonia Detection · AAI-540-02 · Group 4

End-to-end data preparation for the pneumonia CNN: walk raw S3 images → build a unified per-image manifest with authoritative labels → preprocess each image (DICOM/JPEG → grayscale → CLAHE → resize → PNG) → publish manifest CSV → register in SageMaker Feature Store → register in Athena for SQL exploration.

**Five linear sections, single source of truth:**
1. **Setup** — imports + S3 client + bucket-from-config.
2. **Build Metadata** — walk `raw-images/` in S3, derive labels (RSNA from CSV labels, Kermany from folder name), keep `df_metadata` in memory.
3. **Preprocess Images** — `IS_DATA_OWNER`-gated loop: read raw image, CLAHE + resize + PNG, write to S3, build the final manifest with `preprocessed_s3_key` (full `s3://` URI), `pixel_mean`, `pixel_std`, etc. Publishes the canonical `image_metadata.csv` to S3.
4. **Feature Store** — ingest the manifest into SageMaker Feature Store (online + offline stores).
5. **Athena Catalog** — register the published CSV as an external Athena table for SQL queries.

**Differs from earlier drafts:**
- No mid-flow Athena round-trip. The Athena table is registered ONCE at the end against the final preprocessed schema. No more `s3_key` vs `raw_s3_key` schema drift.
- `preprocessed_s3_key` stores the **full s3:// URI**, not a bare key. Self-contained — downstream readers (cicd-pipeline, monitoring) don't need bucket context.
- `IS_DATA_OWNER` defaults to `False` for safety. Cold runs walk the metadata and verify schema; the heavy preprocess + publish only fires when explicitly enabled.
- EDA moved to its own notebook (`eda.ipynb`) so this one is purely a build pipeline.


## Step 1 · Setup

Pin SageMaker + AWS data libs, import everything once, instantiate the boto3 / SageMaker / Athena clients.

In [ ]:
# Pinned versions match what the CI/CD pipeline notebook uses.
%pip uninstall sagemaker -y -q
%pip install "sagemaker>=2.0,<3.0" pyathena awswrangler "boto3>1.17.21" pydicom opencv-python-headless -q

In [ ]:
import io
import json
import time
from datetime import datetime

import awswrangler as wr
import boto3
import cv2
import numpy as np
import pandas as pd
import pydicom
import sagemaker
from pyathena import connect
from sagemaker.feature_store.feature_group import FeatureGroup

from config import BUCKET_NAME, RAW_IMAGE_FOLDER

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
default_bucket = sess.default_bucket()
s3 = boto3.client("s3")

bucket = BUCKET_NAME
raw_prefix = RAW_IMAGE_FOLDER
rsna_labels_prefix = "raw-metadata/rsna"          # populated by data-setup.ipynb §5.4
preprocessed_prefix = "preprocessed-images"        # written by §3 below
metadata_prefix = "pneumonia-project/metadata"     # canonical manifest CSV lives here
manifest_key = f"{metadata_prefix}/image_metadata.csv"
manifest_uri = f"s3://{bucket}/{manifest_key}"

database_name = "pneumonia_db"
table_name = "image_metadata"
athena_staging = f"s3://{default_bucket}/athena/staging"
conn = connect(region_name=region, s3_staging_dir=athena_staging)

print(f"Region:               {region}")
print(f"Role:                 {role}")
print(f"Project bucket:       s3://{bucket}/")
print(f"Default bucket:       s3://{default_bucket}/")
print(f"Canonical manifest:   {manifest_uri}")

## Step 2 · Build the per-image metadata DataFrame

Two reads, then one in-memory join:

1. **RSNA labels** — pull `stage_2_train_labels.csv` and `stage_2_sample_submission.csv` from S3 (uploaded by `data-setup.ipynb` §5.4). The two CSVs together cover every RSNA `patientId`. We collapse them to a single `patientId → Target` map.
2. **S3 walk** — list every image under `raw-images/`. For each:
   - RSNA (`.dcm`): label comes from the map above (authoritative — Kaggle's competition labels).
   - Kermany (`.jpeg`): label comes from the S3 folder name (Kermany has no labels CSV — folder IS the source of truth).

Result: `df_metadata` with `image_id`, `s3_key` (raw image), `source`, `file_type`, `label_int`, `label`, `file_size`. This is the canonical pre-preprocessing view of the dataset.

### 2.1 RSNA labels → patient ID lookup map

In [ ]:
rsna_train = wr.s3.read_csv(f"s3://{bucket}/{rsna_labels_prefix}/stage_2_train_labels.csv")
rsna_sub   = wr.s3.read_csv(f"s3://{bucket}/{rsna_labels_prefix}/stage_2_sample_submission.csv")
rsna_df = pd.concat([rsna_train, rsna_sub], ignore_index=True)

# A patient can have multiple bounding-box rows; the Target column is the same for all rows
# of a given patient, so drop_duplicates collapses to one (patientId, Target) tuple each.
rsna_label_map = dict(zip(rsna_df.patientId, rsna_df.Target.astype(int)))
print(f"RSNA patient IDs with labels: {len(rsna_label_map)}")

### 2.2 Walk `raw-images/` and assemble `df_metadata`

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
rows = []

for page in paginator.paginate(Bucket=bucket, Prefix=f"{raw_prefix}/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        file_name = key.split("/")[-1]
        ext = file_name.rsplit(".", 1)[-1].lower()
        if ext not in ("jpeg", "jpg", "dcm"):
            continue  # skip checkpoints, README, etc.

        image_id = file_name.rsplit(".", 1)[0]
        if ext == "dcm":
            source = "rsna"
            label_int = rsna_label_map.get(image_id)
        else:
            source = "chest_xray"
            label_int = 0 if "NORMAL" in key else 1

        rows.append({
            "image_id":  image_id,
            "s3_key":    key,
            "file_name": file_name,
            "file_type": "dcm" if ext == "dcm" else "jpeg",
            "source":    source,
            "label_int": label_int,
            "file_size": obj["Size"],
        })

df_metadata = pd.DataFrame(rows)
before = len(df_metadata)
df_metadata = df_metadata.dropna(subset=["label_int"]).copy()
df_metadata["label_int"] = df_metadata["label_int"].astype(int)
df_metadata["label"] = df_metadata["label_int"].map({0: "NORMAL", 1: "PNEUMONIA"})

print(f"Total images scanned: {before}")
print(f"Dropped {before - len(df_metadata)} rows with missing RSNA labels")
print(f"Remaining: {len(df_metadata)}")
print(f"Class counts: {df_metadata['label'].value_counts().to_dict()}")
df_metadata.head()

## Step 3 · Preprocess images and publish the manifest

For each raw image:
1. Read from S3 (DICOM via `pydicom`, JPEG via `cv2`).
2. Min-max normalize to uint8 (collapses 12/16-bit DICOM to 8-bit).
3. CLAHE contrast enhancement (clipLimit=2.0, tile 8×8).
4. Resize to 512×512.
5. Encode as PNG, write to `s3://{bucket}/preprocessed-images/<LABEL>/<image_id>.png`.
6. Compute pixel_mean / pixel_std.
7. Append manifest row with **full `s3://` URI** as `preprocessed_s3_key`.

After the loop completes (33K images → ~20–30 min on the SageMaker Studio kernel), republish `image_metadata.csv` to S3.

**Gated behind `IS_DATA_OWNER`.** Default is `False` so cold-runners and graders can re-execute the notebook safely. Flip to `True` once when the bucket needs to be populated; the resulting CSV becomes the source of truth for the cicd-pipeline notebook.

### 3.1 Preprocessing helpers

In [ ]:
IMG_SIZE = (512, 512)
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_image_from_s3(s3_key):
    """Read a single raw image from S3 and return it as a numpy array (grayscale or pixel_array)."""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()
    if s3_key.endswith(".dcm"):
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        return ds.pixel_array
    img_array = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

def preprocess_image(img):
    """Apply min-max normalize → CLAHE → resize. Returns uint8 array of shape IMG_SIZE."""
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = CLAHE.apply(img)
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    return img

# Smoke test on one image.
test_row = df_metadata.iloc[0]
raw = load_image_from_s3(test_row["s3_key"])
out = preprocess_image(raw)
print(f"Raw: {raw.shape}, dtype={raw.dtype}  →  Preprocessed: {out.shape}, dtype={out.dtype}")

### 3.2 Preprocessing loop (data-owner gated)

Flip `IS_DATA_OWNER = True` to run. Writes ~33K PNGs to `s3://{bucket}/preprocessed-images/` (~1 GB total) and republishes the canonical `image_metadata.csv`.

In [ ]:
# Cold-run-safe default. Flip to True once to populate the bucket; flip back when done.
IS_DATA_OWNER = False

In [ ]:
if IS_DATA_OWNER:
    manifest_rows = []
    errors = 0
    n = len(df_metadata)
    print(f"Preprocessing {n} images → s3://{bucket}/{preprocessed_prefix}/")

    REDRAW_EVERY = 100
    for i, (_, row) in enumerate(df_metadata.iterrows(), start=1):
        try:
            raw_img = load_image_from_s3(row["s3_key"])
            processed = preprocess_image(raw_img)

            # Derive folder + manifest label from label_int (TINYINT — can't silently corrupt).
            label_str = {0: "NORMAL", 1: "PNEUMONIA"}[int(row["label_int"])]
            key = f"{preprocessed_prefix}/{label_str}/{row['image_id']}.png"
            _, buf = cv2.imencode(".png", processed)
            s3.put_object(Bucket=bucket, Key=key, Body=buf.tobytes())

            manifest_rows.append({
                "image_id":            str(row["image_id"]),
                "raw_s3_key":          str(row["s3_key"]),
                "preprocessed_s3_key": f"s3://{bucket}/{key}",       # full URI — self-contained
                "label":               label_str,
                "label_int":           int(row["label_int"]),
                "source":              str(row["source"]),
                "file_type":           str(row["file_type"]),
                "pixel_mean":          round(float(np.mean(processed)), 4),
                "pixel_std":           round(float(np.std(processed)), 4),
                "img_height":          int(IMG_SIZE[1]),
                "img_width":           int(IMG_SIZE[0]),
                "event_time":          datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
        except Exception as e:
            errors += 1
            print(f"\n  [{i}] {type(e).__name__}: {e} on {row.get('s3_key', '<missing>')}")
            continue

        if i % REDRAW_EVERY == 0 or i == n:
            print(f"\r  [{i}/{n}] {i/n:6.1%}", end="", flush=True)

    print()
    df_manifest = pd.DataFrame(manifest_rows)
    print(f"Done: preprocessed {len(df_manifest)} images ({errors} errors).")
else:
    print("IS_DATA_OWNER=False — skipping preprocessing. "
          "Will rebuild df_manifest from the published manifest in §3.3 below.")

### 3.3 Publish the manifest CSV (and load it if we skipped §3.2)

Two paths:
- **Owner path:** `df_manifest` is in memory from §3.2; publish it to S3.
- **Cold-runner path:** read the previously-published CSV from S3 so the rest of the notebook works.

Either way, the post-condition is: `df_manifest` is in memory **and** `s3://{bucket}/{manifest_key}` exists with the canonical schema.

In [ ]:
if IS_DATA_OWNER and "df_manifest" in dir():
    df_manifest.to_csv("image_metadata.csv", index=False)
    s3.upload_file("image_metadata.csv", bucket, manifest_key)
    print(f"Published {manifest_uri}")
else:
    # Cold-runner: pull whatever the owner most recently published.
    df_manifest = wr.s3.read_csv(manifest_uri)
    print(f"Loaded df_manifest from {manifest_uri} ({len(df_manifest)} rows).")

# Schema sanity check — fails loudly if the manifest is stale / corrupted.
required = {"image_id", "raw_s3_key", "preprocessed_s3_key", "label", "label_int", "source"}
missing = required - set(df_manifest.columns)
if missing:
    raise ValueError(f"df_manifest is missing required columns: {missing}")

expected_labels = {"NORMAL", "PNEUMONIA"}
unexpected = set(df_manifest["label"].dropna().astype(str).unique()) - expected_labels
if unexpected:
    print(f"WARN: unexpected label values {sorted(unexpected)[:3]}... — rebuilding label from label_int.")
    df_manifest["label"] = df_manifest["label_int"].map({0: "NORMAL", 1: "PNEUMONIA"})

assert df_manifest["label"].isin(expected_labels).all(), "manifest has rows with neither NORMAL nor PNEUMONIA"
print(f"df_manifest is healthy. Class counts: {df_manifest['label'].value_counts().to_dict()}")
df_manifest.head(3)

## Step 4 · SageMaker Feature Store

Register the manifest as a Feature Group. Two stores get populated:
- **Offline store** (S3 + Glue) — queryable via Athena, used for training data discovery.
- **Online store** (DynamoDB-backed) — low-latency lookup for per-image inference enrichment.

We register the full manifest schema (not just engineered features) because the CNN learns its own features; the "feature store" here is really a queryable lookup of `image_id → preprocessed_s3_key + label + pixel_stats`.

### 4.1 Coerce dtypes (Feature Store is strict)

In [ ]:
df_fs = df_manifest.copy()
for col in ["image_id", "raw_s3_key", "preprocessed_s3_key", "label", "source", "file_type", "event_time"]:
    df_fs[col] = df_fs[col].astype(str)
for col in ["label_int", "img_height", "img_width"]:
    df_fs[col] = df_fs[col].astype(int)
for col in ["pixel_mean", "pixel_std"]:
    df_fs[col] = df_fs[col].astype(float)
if "event_time" not in df_fs.columns:
    df_fs["event_time"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    df_fs["event_time"] = df_fs["event_time"].astype(str)
print(df_fs.dtypes)

### 4.2 Define and create the Feature Group

In [ ]:
feature_group_name = "pneumonia-training-manifest"
feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=sess)
feature_group.load_feature_definitions(data_frame=df_fs)

print(f"Feature group: {feature_group_name}")
for fd in feature_group.feature_definitions:
    print(f"  {fd.feature_name}: {fd.feature_type}")

In [ ]:
if IS_DATA_OWNER:
    try:
        feature_group.create(
            s3_uri=f"s3://{default_bucket}/feature-store/",
            record_identifier_name="image_id",
            event_time_feature_name="event_time",
            role_arn=role,
            enable_online_store=True,
        )
        while feature_group.describe().get("FeatureGroupStatus") == "Creating":
            print("  ...creating feature group...")
            time.sleep(5)
        print("Feature group ready.")
    except Exception as e:
        if e.response["Error"]["Code"] == "ResourceInUse":
            print("Feature group already exists. Continuing.")
        else:
            raise
else:
    print("IS_DATA_OWNER=False — skipping feature group create. Assumes the group exists.")

### 4.3 Ingest the manifest

In [ ]:
if IS_DATA_OWNER:
    print(f"Ingesting {len(df_fs)} records into Feature Store...")
    feature_group.ingest(data_frame=df_fs, max_workers=3, wait=True)
    print("Ingest complete.")
else:
    print("IS_DATA_OWNER=False — skipping Feature Store ingest.")

## Step 5 · Register the manifest in Athena

Register the published `image_metadata.csv` as an external Athena table for ad-hoc SQL exploration. Schema is the **post-preprocessing** schema — no early-stage variant, no `s3_key` vs `raw_s3_key` confusion.

In [ ]:
# Make sure the database exists.
pd.read_sql(f"CREATE DATABASE IF NOT EXISTS {database_name}", conn)
# Drop any previously-registered table so the schema below is authoritative.
pd.read_sql(f"DROP TABLE IF EXISTS {database_name}.{table_name}", conn)

# Column order MUST match the CSV's. We control both, so we set them explicitly.
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/{metadata_prefix}/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print(f"Registered {database_name}.{table_name}")

### 5.1 Verify with a few SQL queries

In [ ]:
print("Total rows:")
print(pd.read_sql(f"SELECT COUNT(*) AS n FROM {database_name}.{table_name}", conn))

print("\nLabel distribution:")
print(pd.read_sql(
    f"SELECT label, COUNT(*) AS n FROM {database_name}.{table_name} GROUP BY label",
    conn,
))

print("\nSource × label:")
print(pd.read_sql(
    f"SELECT source, label, COUNT(*) AS n FROM {database_name}.{table_name} "
    f"GROUP BY source, label ORDER BY source, label",
    conn,
))

---

**Done.** Downstream notebooks (`cicd-pipeline.ipynb`, monitoring) read the canonical artifacts from S3:
- Preprocessed PNGs at `s3://{bucket}/preprocessed-images/<LABEL>/<image_id>.png`.
- Manifest CSV at `s3://{bucket}/pneumonia-project/metadata/image_metadata.csv`.
- Feature Store group `pneumonia-training-manifest` (online + offline).
- Athena table `pneumonia_db.image_metadata` for SQL exploration.
